In [10]:
import re
# --- PHASE 0: ERROR HANDLING ---
class ParserError(Exception):
    """Kelas khusus untuk menangani kesalahan saat proses parsing [5]."""
    pass

# --- PHASE 2: AST NODES (REPRESENTASI POHON) ---
class AST:
    pass

class BinOp(AST):
    """Node untuk operasi biner seperti + atau * [2]."""
    def __init__(self, left, op, right):
        self.left = left
        self.op = op
        self.right = right

class Num(AST):
    """Node untuk menyimpan angka [6]."""
    def __init__(self, value):
        self.value = value

class Var(AST):
    """Node untuk menyimpan nama variabel [7]."""
    def __init__(self, name):
        self.name = name

In [11]:
import re

class MiniCompiler:
    def __init__(self, source, env):
        self._tokens = iter(re.findall(r'[a-zA-Z_]\w*|\d+(?:\.\d+)?|[+*/()\-^]', source) + ['?'])
        self._current = None
        self._env = env 
        self._temp_count = 0
        self.advance()

    def advance(self):
        try:
            self._current = next(self._tokens)
        except StopIteration:
            self._current = None

    def expect(self, expected):
        if self._current != expected and not (expected == "ID" and self._current.isalnum()):
            raise ParserError(f"Expected {expected}, found {self._current}")
        token = self._current
        self.advance()
        return token

    def factor(self):
        token = self._current
        if token is not None and token.replace('.', '', 1).isdigit():
            self.advance()
            return Num(float(token) if '.' in token else int(token))
        elif token and token.isalpha():
            if token not in self._env:
                raise ParserError(f"Semantic Error: Undefined variable '{token}'")
            self.advance()
            return Var(token)
        elif token == '(':
            self.advance()
            node = self.expr()
            self.expect(')')
            return node
        raise ParserError(f"Unexpected token: {token}")

    def power(self):
        node = self.factor()
        if self._current == '^': 
            op = self._current
            self.advance()
            right = self.power()  
            node = BinOp(left=node, op=op, right=right)
        return node

    def term(self):
        node = self.power() 
        while self._current in ('*', '/'):
            op = self._current
            self.advance()
            node = BinOp(left=node, op=op, right=self.power()) 
        return node

    def expr(self):
        node = self.term()
        while self._current in ('+', '-'):
            op = self._current
            self.advance()
            node = BinOp(left=node, op=op, right=self.term())
        return node

    def generate_tac(self, node):
        if isinstance(node, Num): return str(node.value)
        if isinstance(node, Var): return node.name
        
        left_val = self.generate_tac(node.left)
        right_val = self.generate_tac(node.right)
        
        self._temp_count += 1
        temp_name = f"t{self._temp_count}"
        print(f"{temp_name} = {left_val} {node.op} {right_val}")
        return temp_name
    
    def evaluate(self, node):
        if isinstance(node, Num):
            return str(node.value), node.value
        if isinstance(node, Var):
            return node.name, self._env[node.name]
        
        left_name,  left_val  = self.evaluate(node.left)
        right_name, right_val = self.evaluate(node.right)
        
        if node.op == '+': result = left_val + right_val
        if node.op == '-': result = left_val - right_val
        if node.op == '*': result = left_val * right_val
        if node.op == '/': result = left_val / right_val
        if node.op == '^': result = left_val ** right_val
        
        self._temp_count += 1
        temp_name = f"t{self._temp_count}"
        print(f"{temp_name} = {left_val} {node.op} {right_val} = {result}")
        return temp_name, result

In [ ]:
source_code = "a ^ 2 + b * c"
symbol_table = {'a': 5, 'b': 10, 'c': 2}

try:
    print(f"Input: {source_code}")
    compiler = MiniCompiler(source_code, symbol_table)
    ast_root = compiler.expr()
    
    print("\n--- Output Three Address Code (TAC) ---")
    compiler.generate_tac(ast_root)
    compiler._temp_count = 0

    print("\n--- Hasil Kalkulasi ---")
    _, hasil = compiler.evaluate(ast_root)
    print(f"Hasil: {hasil}")
    
except Exception as e:
    print(f"Error: {e}")

Input: a + b * c^2

--- Output Three Address Code (TAC) ---
t1 = c ^ 2
t2 = b * t1
t3 = a + t2

--- Hasil Kalkulasi ---
t1 = 2 ^ 2 = 4
t2 = 10 * 4 = 40
t3 = 5 + 40 = 45
Hasil: 45
